In [47]:
import os
import getpass
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Setup Groq API Key
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Initialize Groq LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)

# 3. Load Retrieval Models
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 4. Corpus Setup
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is a bidirectional encoder trained using masked language modelling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization technique used to minimize the loss function.",
    "Neural networks learn by adjusting weights through backpropagation.",
    "Retrieval Augmented Generation combines a retriever with a language model to produce grounded answers.",
    "Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.",
    "Quantization reduces model size by representing weights in lower bit formats such as INT8.",
    "The attention mechanism computes a weighted sum of value vectors based on query-key similarity.",
    "Large language models are trained on massive text corpora to learn general-purpose representations.",
]

# Pre-calculate document vectors
doc_vecs = bi_encoder.encode(corpus, convert_to_numpy=True)
doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

# 5. Define HybridRetriever Class
class HybridRetriever:
    def __init__(self, corpus, doc_vecs, bi_encoder, k=60):
        self.corpus = corpus
        self.doc_vecs = doc_vecs
        self.bi_encoder = bi_encoder
        self.k = k
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

    def retrieve(self, query, top_k=5):
        # BM25
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

        # SBERT
        q_vec = self.bi_encoder.encode([query], convert_to_numpy=True)[0]
        q_vec = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

        # RRF Fusion
        rrf = {d: 1/(self.k+bm25_ranks[d]) + 1/(self.k+sbert_ranks[d]) for d in range(len(self.corpus))}
        final_idx = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [self.corpus[d] for d in final_idx]

hybrid = HybridRetriever(corpus, doc_vecs, bi_encoder)

# 6. Define Chains (Using Groq only)
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Generate a single factual paragraph that would directly answer the question."),
    ("human", "{query}")
])
hyde_chain = hyde_prompt | llm | StrOutputParser()

generation_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the provided context.\n\nContext:\n{context}"),
    ("human", "{question}")
])
gen_chain = generation_prompt | llm | StrOutputParser()

# 7. Helper functions
def rerank(query, candidates):
    pairs = [[query, doc] for doc in candidates]
    scores = cross_encoder.predict(pairs)
    ranked_idx = np.argsort(scores)[::-1][:3]
    return [candidates[i] for i in ranked_idx]

def run_advanced_rag(user_query):
    print(f"\n--- Query: {user_query} ---")

    # Step 1: HyDE Expansion
    hypothetical_doc = hyde_chain.invoke({"query": user_query})
    print(f"[1] HyDE Doc generated...")

    # Step 2: Hybrid Retrieval
    candidates = hybrid.retrieve(hypothetical_doc, top_k=5)

    # Step 3: Re-Ranking
    top_docs = rerank(user_query, candidates)

    # Step 4: Final Generation
    context = "\n\n".join(top_docs)
    answer = gen_chain.invoke({"context": context, "question": user_query})

    print(f"[Result]: {answer}")

# --- EXECUTE ---
run_advanced_rag("How does attention work in language models?")
run_advanced_rag("What is the purpose of backpropagation?")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Query: How does attention work in language models? ---
[1] HyDE Doc generated...
[Result]: The attention mechanism computes a weighted sum of value vectors based on query-key similarity.

--- Query: What is the purpose of backpropagation? ---
[1] HyDE Doc generated...
[Result]: The purpose of backpropagation is to adjust weights in neural networks so they can learn.
